# GENERATE NEW NAMES USING AI. PARTICULARLY MLP
***

Preparing data

In [162]:
with open("data/names.txt","r") as file:
    data = file.read().splitlines()

In [163]:
alphabets = list(set("".join(data)))
alphabets.sort()

stoi = {k:v for v,k in enumerate(alphabets)}
stoi["."] = 26

itos = {v:k for k,v in stoi.items()}

In [164]:
#Iterate through words

for word in data[:1]:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        print(char1,char2,char3,"->",char4)

. . . -> e
. . e -> m
. e m -> m
e m m -> a
m m a -> .


# Creating MLP dataset

In [165]:
import torch
import torch.nn.functional as F
import random

In [166]:
Xs = []
ys = []

random.shuffle(data)
for word in data:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        idx1 = stoi[char1]
        idx2 = stoi[char2]
        idx3 = stoi[char3]
        idx4 = stoi[char4]

        Xs.append([idx1,idx2,idx3])
        ys.append(idx4)



Xs = torch.tensor(Xs)
ys = torch.tensor(ys)

vocab_size = len(stoi)

In [167]:
X_enc = F.one_hot(Xs,num_classes=vocab_size).to(torch.float)
Y_enc = F.one_hot(ys,num_classes=vocab_size).to(torch.float)



In [168]:
tr_idx = int(0.8 * Xs.shape[0])
val_idx = int(0.9 * Xs.shape[0])

X_tr,y_tr = X_enc[:tr_idx], Y_enc[:tr_idx]
X_val, y_val = X_enc[tr_idx:val_idx], Y_enc[tr_idx:val_idx]
X_ts, y_ts = X_enc[val_idx:], Y_enc[val_idx:]

In [169]:
print(f"{X_val.shape= }")
print(f"{X_tr.shape= }")

X_val.shape= torch.Size([22815, 3, 27])
X_tr.shape= torch.Size([182516, 3, 27])


In [170]:
feature_space = 10
batch_size = 10
h_in = feature_space * 3
h_out = 200

Emb = torch.randn((vocab_size,feature_space))
W1 = torch.randn((h_in,h_out))
b1 = torch.randn((1,h_out))

W2 = torch.randn((h_out,vocab_size)) * 0.01
b2 = torch.randn((1,vocab_size)) * 0.0001


parameters = [Emb,W1,b1,W2,b2]

for p in parameters:
    p.requires_grad = True

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))

X_enc[batch_idx].shape

torch.Size([10, 3, 27])

In [171]:
print(f"Size of model : {sum(p.numel() for p in parameters)}")

Size of model : 11897


In [172]:
# forward pass

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))
X = X_enc[batch_idx]
y = Y_enc[batch_idx]
X_emb = X @ Emb
h_preact = X_emb.view((-1,h_in)) @ W1  + b1
h = torch.tanh(h_preact)

o = h @ W2 + b2
loss = F.cross_entropy(o,y)

In [173]:
loss.item()

3.365408420562744

In [174]:
#Full Training

epochs = 100000
log_every = 5000
alpha = 0.01
batch_size = 512

for epoch in range(epochs):
    batch_idx = torch.randint(0,X_tr.shape[0],(batch_size,))
    X = X_tr[batch_idx]
    y = y_tr[batch_idx]
    X_emb = X @ Emb
    h_preact = X_emb.view((-1,h_in)) @ W1  + b1
    h = torch.tanh(h_preact)

    o = h @ W2 + b2
    loss = F.cross_entropy(o,y)

    # Zero grad

    for p in parameters:
        p.grad = None

    loss.backward()

    # Update Weights

    for p in parameters:
        p.data -= alpha * p.grad

    if epoch % log_every == 0:
        print(f"Epoch : {epoch} Loss : {loss.item()} ")

    


Epoch : 0 Loss : 3.3076045513153076 
Epoch : 5000 Loss : 2.33402156829834 
Epoch : 10000 Loss : 2.358863353729248 
Epoch : 15000 Loss : 2.2552316188812256 
Epoch : 20000 Loss : 2.2397284507751465 
Epoch : 25000 Loss : 2.312209129333496 
Epoch : 30000 Loss : 2.243795394897461 
Epoch : 35000 Loss : 2.2856626510620117 
Epoch : 40000 Loss : 2.075923442840576 
Epoch : 45000 Loss : 2.251664876937866 
Epoch : 50000 Loss : 2.2894411087036133 
Epoch : 55000 Loss : 2.1135098934173584 
Epoch : 60000 Loss : 2.120866060256958 
Epoch : 65000 Loss : 2.172736167907715 
Epoch : 70000 Loss : 2.218557596206665 
Epoch : 75000 Loss : 2.258523941040039 
Epoch : 80000 Loss : 2.223389148712158 
Epoch : 85000 Loss : 2.0507028102874756 
Epoch : 90000 Loss : 2.2803800106048584 
Epoch : 95000 Loss : 2.2316393852233887 


In [175]:
# Evaluate on val set


X_val_emb = X_tr @ Emb
h_preact = X_val_emb.view((-1,h_in)) @ W1 + b1
h = torch.tanh(h_preact)
o = h @ W2 + b2
loss = F.cross_entropy(o,y_tr)

print(f"Evaluation on val set : Loss : {loss.item()}")

Evaluation on val set : Loss : 2.1682913303375244
